# Data Loading

Loads the headspace VNA recordings, parses the sweeps, and writes a clean frame
(`processed_data.pkl`) for the downstream notebooks.

The first 10% of sweeps in each session are discarded. When a
sample is first placed in the chamber, the headspace near the antenna takes time
to equilibrate.

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import bisect

from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Plotting style
plt.rcParams.update({
    'font.size': 10,
    'figure.figsize': (10, 5)
})

# Constants
DATA_DIR = '../data'
OUTPUT_FILE = 'processed_data.pkl'

In [ ]:
def load_and_process_csv(filepath, target_date):
    """
    Parses a single headspace CSV, calculates the timedelta class, 
    and flattens each valid frequency sweep into a single feature vector.
    """
    filename = os.path.basename(filepath)
    time_str = filename.replace('headspace_data_', '').replace('.csv', '')
    csv_timestamp = pd.to_datetime(time_str, format='%Y%m%d_%H%M%S')
    
    print(filename)
    
    # Calculate the timedelta class (time until the best before date)
    time_delta = target_date - csv_timestamp
    timedelta_days = time_delta.total_seconds() / (24 * 3600.0)
    
    df = pd.read_csv(filepath)
    
    # Group by sweeps: Frequency resets to 1MHz at the start of a new sweep
    df['sweep_id'] = (df['Frequency_Hz'].diff() < 0).cumsum()
    expected_length = df['sweep_id'].value_counts().mode()[0]
    
    sweeps_data = []
    for sweep_id, sweep_df in df.groupby('sweep_id'):
        if len(sweep_df) != expected_length:
            continue # Skip aborted/incomplete sweeps
            
        s11_real = sweep_df['S11_Real'].values
        s11_imag = sweep_df['S11_Imag'].values
        s22_real = sweep_df['S22_Real'].values
        s22_imag = sweep_df['S22_Imag'].values
        
        # Split into Sensor A (S11) and Sensor B (S22)
        sweep_features_a = np.concatenate([s11_real, s11_imag])
        sweep_features_b = np.concatenate([s22_real, s22_imag])
        
        # We only need the inner boundaries for bisect
        boundaries = [-3, -2, -1, 0, 1, 2, 3]
        labels = ['0', '1', '2', '3', '4', '5', '6', '7']

        # bisect_right perfectly mimics pd.cut(right=False)
        index = bisect.bisect_right(boundaries, timedelta_days)
        timedelta_class = labels[index]
        
        
        sweeps_data.append({
            'filename': filename,
            'target_class_date': target_date.strftime('%Y%m%d'), # Keep track of which folder it came from
            'csv_timestamp': csv_timestamp,
            'timedelta_days': timedelta_days,
            'timedelta_class': timedelta_class,
            'sweep_id': sweep_id,
            'features_a': sweep_features_a,
            'features_b': sweep_features_b
        })
        
    return pd.DataFrame(sweeps_data)

In [ ]:
all_data = []

# Scan the data directory for all class folders
for entry in os.scandir(DATA_DIR):
    if entry.is_dir():
        folder_name = entry.name
        try:
            target_date = pd.to_datetime(folder_name, format='%Y%m%d')
        except ValueError:
            continue # Skip non-date folders
            
        search_pattern = os.path.join(entry.path, 'headspace_data_*.csv')
        csv_files = sorted(glob.glob(search_pattern))
        
        for filepath in csv_files:
            df_sweeps = load_and_process_csv(filepath, target_date)
            
            # burn-in discard: early post-loading sweeps are pre-equilibrium and
            # inflate within-session variance, so drop the first 10% per session.
            df_sweeps = df_sweeps.iloc[int(len(df_sweeps)*0.1):].copy()
            
            all_data.append(df_sweeps)

if not all_data:
    print("No valid sweeps found in any directory.")
else:
    df_all = pd.concat(all_data, ignore_index=True)
    print(f"Total completed sweeps extracted across all classes: {len(df_all)}")
    
    # export for downstream notebooks
    df_all.to_pickle(OUTPUT_FILE)
    print(f"Successfully saved dataset to {OUTPUT_FILE}")
    display(df_all.head())